In [1]:
!pip install transformers datasets peft accelerate scikit-learn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/207.5 MB 216.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 15.1 MB/s eta 0:00:00
ERROR: THESE PACKAGES DO NOT MATCH THE HASHES FROM THE REQUIREMENTS FILE. If you have updated the package versions, please update the hashes. Otherwise, examine the package contents carefully; someone 

In [2]:
from google.colab import files

print("🔼 Upload LoRA model zip (e.g., lora_model.zip):")
lora_zip = files.upload()

print("🔼 Upload train.csv")
csv_files = files.upload()


🔼 Upload LoRA model zip (e.g., lora_model.zip):


Saving lora_model.zip to lora_model.zip
🔼 Upload train.csv, val.csv, and test.csv:


Saving train.csv to train.csv


In [3]:
import zipfile
import os

with zipfile.ZipFile("lora_model.zip", 'r') as zip_ref:
    zip_ref.extractall("lora_model")


In [6]:
peft_path = "lora_model/lora-bert-model"  # ✅ correct subdirectory containing adapter_config.json

from peft import PeftConfig, PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load adapter config from correct folder
peft_config = PeftConfig.from_pretrained(peft_path)

# Load base model
base_model = AutoModelForSequenceClassification.from_pretrained(peft_config.base_model_name_or_path)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, peft_path)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)

model.eval()


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default

In [8]:
import pandas as pd

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")

In [19]:
@torch.no_grad()
def get_embeddings(texts, batch_size=16):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # ✅ Use encoder directly (not classification head)
        outputs = model.base_model.bert(**inputs, output_hidden_states=True, return_dict=True)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]  # CLS token
        embeddings.append(cls_embeddings.cpu())

    return torch.cat(embeddings, dim=0)



In [23]:
# Compute prototype embeddings from training data
train_embeddings = get_embeddings(train_df["text"].tolist())
train_labels = train_df["label"].values

# Create prototypes by averaging embeddings for each class
pos_proto = train_embeddings[train_labels == 1].mean(dim=0)
neg_proto = train_embeddings[train_labels == 0].mean(dim=0)


In [24]:
from sklearn.metrics.pairwise import cosine_similarity

pos_sim = cosine_similarity(test_embeddings, pos_proto.unsqueeze(0))
neg_sim = cosine_similarity(test_embeddings, neg_proto.unsqueeze(0))
preds = (pos_sim > neg_sim).astype(int).flatten()



In [30]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 🔧 Set how many neighbors to display
k = 3

# For each test sample
for i in range(5):  # limit to 5 test samples for display
    test_vec = test_embeddings[i].unsqueeze(0)  # shape (1, hidden_dim)

    # Compute cosine similarity to all training samples
    sims = cosine_similarity(test_vec, train_embeddings)[0]  # shape (train_size,)

    # Get top-k most similar training indices
    top_k_idx = sims.argsort()[-k:][::-1]  # descending order

    print(f"\n🧠 Test Sample {i+1}:")
    print(f"Text: {test_df['text'].iloc[i]}")
    print(f"True Label: {'Positive' if test_labels[i]==1 else 'Negative'}")

    for rank, idx in enumerate(top_k_idx):
        label = 'Positive' if train_labels[idx] == 1 else 'Negative'
        sim_score = sims[idx]
        print(f"  → Neighbor {rank+1}: [{label}] Similarity = {sim_score:.4f}")
        print(f"    Text: {train_df['text'].iloc[idx][:200]}{'...' if len(train_df['text'].iloc[idx]) > 200 else ''}")



🧠 Test Sample 1:
Text: there may not be a critic alive who harbors as much affection for shlock monster movies as i do .
i delighted in the sneaky - smart entertainment of ron underwood 's big - underground - worm yarn tremors ; i even giggled at last year 's critically - savaged big - underwater - snake yarn anaconda .
something about these films causes me to lower my inhibitions and return to the saturday afternoons of my youth , spent in the company of ghidrah , the creature from the black lagoon and the blob .
deep rising , a big - undersea - serpent yarn , does n't quite pass the test .
sure enough , all the modern monster movie ingredients are in place : a conspicuously multi - ethnic / multi - national collection of bait .
.. excuse me , characters ; an isolated location , here a derelict cruise ship in the south china sea ; some comic relief ; a few cgi - enhanced gross - outs ; and at least one big explosion .
there are too - cheesy - to - be - accidental elements , like a sl

In [31]:
from sklearn.metrics.pairwise import cosine_similarity

def explain_with_neighbors(test_embeddings, test_texts, test_labels,
                           train_embeddings, train_texts, train_labels,
                           k=3, target_top_label=1, samples_to_show=3):
    shown = 0
    for i in range(len(test_embeddings)):
        test_vec = test_embeddings[i].unsqueeze(0)
        sims = cosine_similarity(test_vec, train_embeddings)[0]
        top_k_idx = sims.argsort()[-k:][::-1]

        # Check the label of the top neighbor
        top_label = train_labels[top_k_idx[0]]
        if top_label != target_top_label:
            continue

        print(f"\n🧠 Test Sample {i+1} — Nearest Match is {'Positive' if top_label==1 else 'Negative'}")
        print(f"Text: {test_texts[i]}")
        print(f"True Label: {'Positive' if test_labels[i]==1 else 'Negative'}")

        for rank, idx in enumerate(top_k_idx):
            neighbor_label = 'Positive' if train_labels[idx] == 1 else 'Negative'
            sim_score = sims[idx]
            print(f"  → Neighbor {rank+1}: [{neighbor_label}] Similarity = {sim_score:.4f}")
            print(f"    Text: {train_texts[idx][:200]}{'...' if len(train_texts[idx]) > 200 else ''}")

        shown += 1
        if shown >= samples_to_show:
            break

# Prepare variables
test_texts = test_df["text"].tolist()
train_texts = train_df["text"].tolist()

# 🔹 Show 3 test examples whose nearest neighbor is Positive
print("\n🌟 Showing 3 samples with Positive nearest neighbor:")
explain_with_neighbors(test_embeddings, test_texts, test_labels,
                       train_embeddings, train_texts, train_labels,
                       k=3, target_top_label=1, samples_to_show=3)

# 🔸 Show 3 test examples whose nearest neighbor is Negative
print("\n🔻 Showing 3 samples with Negative nearest neighbor:")
explain_with_neighbors(test_embeddings, test_texts, test_labels,
                       train_embeddings, train_texts, train_labels,
                       k=3, target_top_label=0, samples_to_show=3)



🌟 Showing 3 samples with Positive nearest neighbor:

🧠 Test Sample 2 — Nearest Match is Positive
Text: renee zellweger stars as sonia , a young jewish wife and mother frustrated by the constraints of her hasidic community in brooklyn .
her husband ( glenn fitzgerald ) is a religious scholar whose all - in - a - day's - work attitude on sex fails to tame the " fire " she feels within , as so she confesses to the rebbe ( after hearing her fiery confession , the rebbe suddenly gets frisky with his pleasantly surprised wife -- and dies the next morning ) .
sensing her frustration , her husband 's brother ( christopher eccleston ) gives her a job in his jewelry brokering business in exchange for raw , passionless sex that just fans sonia 's still - burning flame .
on the job , sonia befriends ramon ( allen payne ) , a cool blast of hunky puerto rican water who does his own jewelry designs when not working as a grunt in an upscale jewelry store .
can fire - taming be far be that far behind 

In [37]:
from sklearn.metrics.pairwise import cosine_similarity

def get_nn_explanations(test_embeddings, test_texts, test_labels,
                        train_embeddings, train_texts, train_labels,
                        preds, target_label, max_samples=8, k=3):
    explanations = []
    count = 0
    for i in range(len(test_embeddings)):
        if preds[i] != target_label or test_labels[i] != target_label:
            continue  # only consider correctly predicted class

        sims = cosine_similarity(test_embeddings[i].unsqueeze(0), train_embeddings)[0]
        top_k_idx = sims.argsort()[-k:][::-1]

        entry = {
            "test_text": test_texts[i],
            "true_label": "Positive" if test_labels[i]==1 else "Negative",
            "pred_label": "Positive" if preds[i]==1 else "Negative",
            "neighbors": []
        }

        for rank, idx in enumerate(top_k_idx):
            entry["neighbors"].append({
                "neighbor_rank": rank+1,
                "label": "Positive" if train_labels[idx]==1 else "Negative",
                "similarity": float(sims[idx]),
                "text": train_texts[idx][:300] + ("..." if len(train_texts[idx]) > 300 else "")
            })

        explanations.append(entry)
        count += 1
        if count >= max_samples:
            break
    return explanations


In [38]:
# Ensure test embeddings, labels, texts, and train counterparts are all defined
test_texts = test_df["text"].tolist()
train_texts = train_df["text"].tolist()

# Generate explanations
positive_expls = get_nn_explanations(test_embeddings, test_texts, test_labels,
                                     train_embeddings, train_texts, train_labels,
                                     preds, target_label=1, max_samples=8)

negative_expls = get_nn_explanations(test_embeddings, test_texts, test_labels,
                                     train_embeddings, train_texts, train_labels,
                                     preds, target_label=0, max_samples=8)

all_expls = positive_expls + negative_expls


In [40]:
import pandas as pd

def flatten_explanations_to_df(expls):
    rows = []
    for ex in expls:
        row = {
            "test_text": ex["test_text"],
            "true_label": ex["true_label"],
            "pred_label": ex["pred_label"],
        }
        for i, neighbor in enumerate(ex["neighbors"]):
            row[f"neighbor_{i+1}_label"] = neighbor["label"]
            row[f"neighbor_{i+1}_similarity"] = neighbor["similarity"]
            row[f"neighbor_{i+1}_text"] = neighbor["text"]
        rows.append(row)
    return pd.DataFrame(rows)


In [41]:
df_expls = flatten_explanations_to_df(all_expls)
df_expls.to_csv("prototype_nn_explanations.csv", index=False)


In [43]:
from google.colab import files
files.download("prototype_nn_explanations.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>